# 10. Entrenamiento final y versionado de artefactos

**Fases del guía metodológica cubiertas: 16 (Entrenamiento final y versionado de artefactos)**

> Regla central: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.



## 16.1 Decisiones congeladas

- Features: las 37 columnas (29 originales + 8 derivadas candidatas) — `configs/final_features.json`.
- Preprocessing: pipeline ColumnTransformer (OneHot + Ordinal + mediana) — `src/models/train_model.py`.
- Modelo: **RandomForest** tuneado con hiperparámetros de `configs/best_params_rf.json` (fase 14).
- Umbral: se ajustará sobre validation en esta fase (bloqueado en metadatos).
- Política de abstención: zona de baja confianza [0.30, 0.60) -> revisión humana.

## 16.2 Reentrenamiento

La selección ya terminó -> reentrenamos con **train + validation** (80 % de los datos).

### 16.2.1 Preparación

Cargamos los conjuntos y los mejores parámetros de RandomForest. La semilla, las features
y el preprocesado quedan congelados: a partir de aquí **no se modifica nada** de la
configuración, solo se entrena el pipeline final.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
from src.models.train_model import train_final

d = load_processed()
Xtr = add_domain_features(d["X_train"]); ytr = d["y_train"]
Xva = add_domain_features(d["X_val"]);   yva = d["y_val"]
Xte = add_domain_features(d["X_test"]);  yte = d["y_test"]

best = json.loads((ROOT / "configs" / "best_params_rf.json").read_text(encoding="utf-8"))
print("Hiperparámetros congelados:", best)


Hiperparámetros congelados: {'n_estimators': 550, 'max_depth': 9, 'min_samples_leaf': 3, 'min_samples_split': 2, 'max_features': 0.5284109559755278, 'class_weight': 'balanced', 'random_state': 42}



### 16.2.2 Entrenamiento con train + validation

Construimos el modelo final con los parámetros congelados y lo entrenamos con
**train + validation** (529 registros), como recomienda el índice (fase 16.2: la fase de
selección ya terminó). El pipeline completo (preprocessor + modelo) se ajusta una única
vez; el test no participa en ningún `fit`.

Después ajustamos un **calibrador sigmoidal (Platt)** sobre validation: transforma las
probabilidades crudas del RandomForest en probabilidades bien calibradas. Se eligió la
calibración sigmoidal frente a la isotónica porque, con el dataset ampliado de 662
alumnos, la isotónica sobreajusta (comprime las probabilidades en 3 escalones y empeora
el ECE a 0.23), mientras que la sigmoidal reparte las probabilidades por todo el rango y
logra ECE ~ 0.13 sin alterar el ranking (el ROC-AUC es invariante a transformaciones
monótonas). El calibrador se guarda como artefacto `models/final_calibrator.joblib` y la
API lo aplicará en producción.


In [2]:

# Construimos el modelo final con los parámetros congelados y reentrenamos con train+val
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd
from src.models.train_model import make_pipeline, get_preprocessor

final_model = RandomForestClassifier(**best)
pipeline = make_pipeline(final_model)

X_fit = pd.concat([Xtr, Xva], axis=0)
y_fit = pd.concat([ytr, yva], axis=0)
pipeline.fit(X_fit, y_fit)
print("Entrenado con", len(X_fit), "registros (train + validation)")

# Recalibración sigmoidal (Platt) sobre validation: robusta con muestra amplia
p_val = pipeline.predict_proba(Xva)[:, 1]
calibrator = LogisticRegression(max_iter=1000)
calibrator.fit(p_val.reshape(-1, 1), yva)
print("Calibrador sigmoidal (Platt) ajustado sobre validation.")


Entrenado con 529 registros (train + validation)
Calibrador sigmoidal (Platt) ajustado sobre validation.



### 16.2.3 Guardado de artefactos y umbral de coste

Guardamos el pipeline (modelo + preprocessor) en `models/final_model.joblib`. Después
calculamos el **umbral óptimo por coste** (FP=1, FN=2) sobre **validation** y lo
congelamos en `models/final_model_metadata.json` junto con la lista de features, los
hiperparámetros y la semilla. Estos metadatos son el contrato que usará la API (fase 21)
para decidir la clase final a partir de la probabilidad.


In [3]:

# Guardar artefactos (fase 16.3)
import joblib
(ROOT / "models").mkdir(exist_ok=True)
joblib.dump(pipeline, ROOT / "models" / "final_model.joblib")
joblib.dump(calibrator, ROOT / "models" / "final_calibrator.joblib")

# Umbral de coste sobre validation (fase 6.3) - se congela en metadatos
from src.evaluation.metrics import find_optimal_threshold
proba_val = calibrator.predict_proba(pipeline.predict_proba(Xva)[:, 1].reshape(-1, 1))[:, 1]
umbral = find_optimal_threshold(yva, proba_val)["optimal_threshold"]
print("Umbral óptimo por coste (validation, calibrado):", round(umbral, 3))

metadata = {
    "model_name": "RandomForest",
    "params": best,
    "features": list(X_fit.columns),
    "threshold": float(umbral),
    "random_state": 42,
    "n_train": int(len(X_fit)),
    "dataset_version": "uci-student-performance-2008",
    "target_definition": "Walc >= 3 (consumo alto fin de semana)",
    "preprocessing": "ColumnTransformer(OneHot + Ordinal + mediana), fit solo con train",
    "calibration": "Sigmoidal (Platt) ajustado sobre validation (dataset ampliado 662 alumnos)",
    "selected_by": "fase 15: Repeated CV 3x5 + validation (mejor equilibrio calidad/estabilidad)",
}
(ROOT / "models" / "final_model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Artefactos guardados en models/")


Umbral óptimo por coste (validation, calibrado): 0.34
Artefactos guardados en models/



### 16.2.4 Registro del experimento final

Añadimos una fila al registro `reports/experiments.csv` documentando el entrenamiento
final (modelo, hiperparámetros principales, umbral y tamaño de entrenamiento). Esto
completa la disciplina experimental: cualquier persona puede reconstruir qué se entrenó,
con qué configuración y cuándo.


In [4]:

# Registro del experimento final
exp_row = f"exp-final,RandomForest,{best['n_estimators']},{best['max_depth']},{best['min_samples_leaf']},{umbral:.3f},{pipeline.__class__.__name__},train+val,{X_fit.shape[0]}"
(ROOT / "reports" / "experiments.csv").write_text(
    (ROOT / "reports" / "experiments.csv").read_text(encoding="utf-8") + exp_row + "\n",
    encoding="utf-8")
print("Registro actualizado.")


Registro actualizado.
